In [ ]:
import ee,geemap

Map = geemap.Map()
Map

In [ ]:
# 1985--2000
five_year = ee.ImageCollection("projects/sat-io/open-datasets/GLC-FCS30D/five-years-map")

# 2000年后
annual = ee.ImageCollection("projects/sat-io/open-datasets/GLC-FCS30D/annual")

In [ ]:
uniqueClasses = [10, 11, 12, 20, 51, 52, 61, 62, 71, 72, 81, 82, 91, 92,120, 121, 122, 130, 140, 150, 152, 153, 181, 182,183, 184, 185, 186, 187, 190, 200, 201, 202, 210, 220, 0]

palette = [
  "#ffff64", "#ffff64", "#ffff00", "#aaf0f0", "#4c7300", "#006400", "#a8c800", "#00a000", 
  "#005000", "#003c00", "#286400", "#285000", "#a0b432", "#788200", "#966400", "#964b00", 
  "#966400", "#ffb432", "#ffdcd2", "#ffebaf", "#ffd278", "#ffebaf", "#00a884", "#73ffdf", 
  "#9ebb3b", "#828282", "#f57ab6", "#66cdab", "#444f89", "#c31400", "#fff5d7", "#dcdcdc", 
  "#fff5d7", "#0046c8", "#ffffff", "#ffffff"
]

def recodeClasses(image):
    reclassed = image.remap(uniqueClasses, ee.List.sequence(1, len(uniqueClasses)))
    return reclassed

### 矢量研究区

In [ ]:
seaside_fishNet = ee.FeatureCollection("projects/ee-2431566134liumonarch/assets/EAAF_seaside_fishNet")
Map.addLayer(seaside_fishNet, {}, 'seaside_fishNet')

In [ ]:
country_fishnet = ee.FeatureCollection("projects/ee-2431566134liumonarch/assets/EAAF_Country_fishNet_diss")
Map.addLayer(country_fishnet, {}, 'country_fishnet')

In [ ]:
fishnet = ee.FeatureCollection("projects/ee-2431566134liumonarch/assets/EAAF_roi_fishnet")
Map.addLayer(fishnet, {}, 'fishnet')

In [ ]:
ramsar = ee.FeatureCollection("projects/ee-2431566134liumonarch/assets/EAAF_Ramsar_sites_diss")
Map.addLayer(ramsar, {}, 'ramsar')

In [ ]:
china_seaside_fishNet_1km = ee.FeatureCollection("projects/ee-2431566134liumonarch/assets/China_seaside_fishNet_1km")
Map.addLayer(china_seaside_fishNet_1km, {}, "china_seaside_fishNet_1km")

In [ ]:
china_seaside_fishNet_5km = ee.FeatureCollection("projects/ee-2431566134liumonarch/assets/China_seaside_fishNet_5km")
Map.addLayer(china_seaside_fishNet_5km, {}, "china_seaside_fishNet_5km")

In [ ]:
import pandas as pd
import geopandas as gpd

In [ ]:
eaaf_gdf = gpd.read_file("EAAF_Ramsar_sites.shp").to_crs("epsg:4326")
eaaf = geemap.gdf_to_ee(eaaf_gdf)
Map.addLayer(eaaf, {}, "eaaf")

### GLC_FCS30D调用

In [ ]:
# 1985--2000
# GLC_1985_image = recodeClasses(five_year.mosaic().select(f'b1'))
# GLC_1985_image = recodeClasses(five_year.mosaic().select(f'b2'))
images = []
images_name = []
for i in [1, 2, 3]:
    image = recodeClasses(five_year.mosaic().select(f'b{i}'))
    images.append(image)
    images_name.append(f"{1985+(i-1)*5}")

# 2000年后
# GLC_2000_image = recodeClasses(annual.mosaic().select(f'b1'))
# GLC_2001_image = recodeClasses(annual.mosaic().select(f'b2'))
for i in range(1, 24):
    image = recodeClasses(annual.mosaic().select(f'b{i}'))
    images.append(image)
    images_name.append(f"{2000+i-1}")

In [ ]:
import os

os.chdir(r"D:\Work\EAAF")

In [ ]:
Map.addLayer(images[0], {'palette': palette}, "GLC_1985")
Map.addLayer(images[4], {'palette': palette}, "GLC_2000")

In [ ]:
image = ee.Image(images).rename(images_name)
image.bandNames()

In [ ]:
roi = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Rectangle([115, 2, 120, 7]), {"name": "A"}),
    ee.Feature(ee.Geometry.Rectangle([111, 0, 116, 5]), {"name": "B"})
])

### 农田和不透水分区统计

In [ ]:
roi = china_seaside_fishNet_1km

In [ ]:
image_Impervious =  image.updateMask(image.eq(30))

eaaf_ramsar_Impervious_count = image_Impervious.reduceRegions(**{
    'reducer': ee.Reducer.count(),
    'collection': roi,
    "scale": 30,
    "crs": "epsg:3857"
})

image_cropland =  image.updateMask(image.gte(1).And(image.lte(4)))

eaaf_ramsar_cropland_count = image_cropland.reduceRegions(**{
    'reducer': ee.Reducer.count(),
    'collection': roi,
    "scale": 30,
    "crs": "epsg:3857"
})

#### 导出至asset

In [ ]:
geemap.ee_export_vector_to_asset(eaaf_ramsar_Impervious_count, "eaaf_ramsar_Impervious_count", "eaaf_ramsar_Impervious_count")

In [ ]:
geemap.ee_export_vector_to_asset(eaaf_ramsar_cropland_count, "eaaf_ramsar_cropland_count", "eaaf_ramsar_cropland_count")

#### 导出至表格

In [ ]:
eaaf_seaside_fishNet_Impervious_count = ee.FeatureCollection("users/2431566134liumonarch/eaaf_seaside_fishNet_Impervious_count")
geemap.ee_to_shp(eaaf_seaside_fishNet_Impervious_count, "eaaf_seaside_fishNet_Impervious_count.shp")

In [ ]:
eaaf_seaside_fishNet_cropland_count = ee.FeatureCollection("users/2431566134liumonarch/eaaf_seaside_fishNet_cropland_count")
geemap.ee_to_shp(eaaf_seaside_fishNet_cropland_count, "eaaf_seaside_fishNet_cropland_count.shp")

In [ ]:
gdf1 = gpd.read_file(f"eaaf_seaside_fishNet_cropland_count.shp")
gdf1.iloc[:, :26] *= 900
gdf1.iloc[:, :26] /= 1000000
gdf1

In [ ]:
gdf1.to_file("EAAF_seaside_croplan_fishNet.shp")

In [ ]:
gdf1 = gdf1.sort_values("name")
gdf1.reset_index(inplace=True)

In [ ]:
columns = ['name'] + images_name
df = gdf1[columns]
df.iloc[:, 1:] *= 900
df.iloc[:, 1:] /= 1000000
df

In [ ]:
df.to_csv("EAAF_Cropland_Area_Country.csv", index=False)

### 湿地分区统计

In [ ]:
image_mangrove =  image.updateMask(image.eq(27))

china_seaside_fishNet_1km_mangrove_count = image_mangrove.reduceRegions(**{
    'reducer': ee.Reducer.count(),
    'collection': roi,
    "scale": 30,
    "crs": "epsg:3857"
})

image_salt =  image.updateMask(image.eq(28))
china_seaside_fishNet_1km_salt_count = image_salt.reduceRegions(**{
    'reducer': ee.Reducer.count(),
    'collection': roi,
    "scale": 30,
    "crs": "epsg:3857"
})

image_tidal =  image.updateMask(image.eq(29))
china_seaside_fishNet_1km_tidal_count = image_tidal.reduceRegions(**{
    'reducer': ee.Reducer.count(),
    'collection': roi,
    "scale": 30,
    "crs": "epsg:3857"
})

image_total = image.updateMask(image.gte(27).And(image.lte(29)))
china_seaside_fishNet_1km_total_count = image_total.reduceRegions(**{
    'reducer': ee.Reducer.count(),
    'collection': roi,
    "scale": 30,
    "crs": "epsg:3857"
})

In [ ]:
geemap.ee_export_vector_to_asset(china_seaside_fishNet_1km_mangrove_count, "china_seaside_fishNet_1km_mangrove_count", "china_seaside_fishNet_1km_mangrove_count")

geemap.ee_export_vector_to_asset(china_seaside_fishNet_1km_salt_count, "china_seaside_fishNet_1km_salt_count", "china_seaside_fishNet_1km_salt_count")

geemap.ee_export_vector_to_asset(china_seaside_fishNet_1km_tidal_count, "china_seaside_fishNet_1km_tidal_count", "china_seaside_fishNet_1km_tidal_count")

geemap.ee_export_vector_to_asset(china_seaside_fishNet_1km_total_count, "china_seaside_fishNet_1km_total_count", "china_seaside_fishNet_1km_total_count")

In [ ]:
china_seaside_fishNet_1km_mangrove_count = ee.FeatureCollection("users/2431566134liumonarch/china_seaside_fishNet_1km_mangrove_count")
geemap.ee_to_shp(china_seaside_fishNet_1km_mangrove_count, "china_seaside_fishNet_1km_mangrove_count.shp")

china_seaside_fishNet_1km_salt_count = ee.FeatureCollection("users/2431566134liumonarch/china_seaside_fishNet_1km_salt_count")
geemap.ee_to_shp(china_seaside_fishNet_1km_salt_count, "china_seaside_fishNet_1km_salt_count.shp")

china_seaside_fishNet_1km_tidal_count = ee.FeatureCollection("users/2431566134liumonarch/china_seaside_fishNet_1km_tidal_count")
geemap.ee_to_shp(china_seaside_fishNet_1km_tidal_count, "china_seaside_fishNet_1km_tidal_count.shp")

china_seaside_fishNet_1km_total_count = ee.FeatureCollection("users/2431566134liumonarch/china_seaside_fishNet_1km_total_count")
geemap.ee_to_shp(china_seaside_fishNet_1km_total_count, "china_seaside_fishNet_1km_total_count.shp")

In [ ]:
t = "total"

In [ ]:
gdf1 = gpd.read_file(f"china_seaside_fishNet_1km_{t}_count.shp")
gdf1

In [ ]:
df_sum = gdf1.iloc[:, :26].sum(axis=1)
gdf2 = gdf1.loc[df_sum > 0, :]
gdf2.iloc[:, :26] *= 900
gdf2.iloc[:, :26] /= 1000000
gdf2

In [ ]:
gdf2.to_file(f"China_fishNet_1km_{t}_noValue.shp")

In [ ]:
gdf1 = gpd.read_file(f"temp/eaaf_country_mangrove_count.shp")
df1 = gdf1.iloc[:, :26]
columns = ["name"] +  df1.columns.to_list()
df2 = gdf1[columns]
df2= df2.astype(object)
df2.iloc[:, 1:] *= 900
df2.iloc[:, 1:] /= 1000000
df2.to_csv("EAAF_Mangrove_Area_Country")

In [ ]:
gdf = gpd.read_file("EAAF_Ramsar_sites_diss.shp") 

In [ ]:
gdf1 = geemap.ee_to_gdf(eaaf_ramsar_tidal_count)

In [ ]:
columns = ['FIRST_v_id', 'FIRST_rams', 'FIRST_offi', 'FIRST_iso3', 'FIRST_coun', 'FIRST_area'] + images_name
columns_r = ['v_idris', 'ramsarid', 'officialna', 'iso3', 'country_en', 'area_off'] + images_name
df = gdf1[columns]
df.columns = columns_r
df.iloc[:, 6:] *= 900
df.iloc[:, 6:] /= 1000000
df.sort_values("ramsarid", inplace=True)
gdf2 = gpd.GeoDataFrame(df, crs=4326, geometry=gdf["geometry"])
gdf2.to_file("EAAF_Tidal_Area_Ramsar.shp")

In [ ]:
df.to_csv("EAAF_Tidal_Area_Ramsar.csv", index=False)

In [ ]:
import pandas as pd
import geopandas as gpd

In [ ]:
t = "mangrove"

In [ ]:
gdf1 = gpd.read_file(f"eaaf_ramsar_{t}_count.shp")
gdf1

In [ ]:
df_sum = gdf1.iloc[:, :26].sum(axis=1)
gdf2 = gdf1.loc[df_sum > 0, :]
gdf2.iloc[:, :26] *= 900
gdf2.iloc[:, :26] /= 1000000
df2 = gdf2.iloc[:, :26]
df2

In [ ]:
from scipy import stats
import pandas as pd

columns = df2.columns
x = [int(i) for i in columns]
slope = []
rvalue = []
def fun(data):
    y = data.values
    OLS = stats.linregress(x, y)
    slope.append(OLS[0])
    rvalue.append(OLS[2]**2)
    return OLS[3]
pvalue = df2.apply(fun, axis=1)

In [ ]:
gdf2["pvalue"] = pvalue
gdf2['slope'] = slope
gdf2['r2'] = rvalue
gdf2.to_file(f"EAAF_{t}_Area_fistNet_noValue.shp")

In [ ]:
gdf3 = gdf2.loc[pvalue < 0.05, :]
gdf3.to_file(f"EAAF_{t}_Area_fistNet_slope.shp")